In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers datasets accelerate scikit-learn pandas


Mounted at /content/drive


In [2]:
# 1. Устанавливаем модуль для поддержки feather (если Colab его попросит)
!pip install -q pyarrow

import pandas as pd

# 2. Читаем feather-файл напрямую с Google Диска
# Замените 'ваш_путь' на реальную папку на Диске, куда закинете файл
df = pd.read_feather('/content/drive/MyDrive/tg_channels.feather')

# 3. Смотрим структуру
print(df.info())
print(df.head(3))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2900 entries, 0 to 2899
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype              
---  ------   --------------  -----              
 0   url      2900 non-null   object             
 1   date     2900 non-null   datetime64[ns, UTC]
 2   content  2852 non-null   object             
 3   label    2900 non-null   object             
dtypes: datetime64[ns, UTC](1), object(3)
memory usage: 90.8+ KB
None
                               url                      date  \
0  https://t.me/s/russianmoda/5961 2023-09-17 17:09:06+00:00   
1  https://t.me/s/russianmoda/5957 2023-09-17 12:09:06+00:00   
2  https://t.me/s/russianmoda/5951 2023-09-17 06:09:06+00:00   

                                             content label  
0  Широкие бедра всегда считались роскошью и прид...  мода  
1  Не знаю как у вас, но у меня ни одна осень не ...  мода  
2  Осенью настроение немножко ухудшается. Этому с...  мода  


In [3]:
# Удаляем строки, где нет текста в content
df = df.dropna(subset=['content']).reset_index(drop=True)

print("Размер датасета после очистки:", df.shape)
print("\nРаспределение классов:")
print(df['label'].value_counts())

Размер датасета после очистки: (2852, 4)

Распределение классов:
label
мода          598
технологии    589
финансы       583
крипта        583
спорт         499
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib

# Переименуем колонку label в target_label, чтобы не путать с полем label в Hugging Face
df = df.rename(columns={'label': 'target_label'})

# Кодируем классы
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['target_label'])

# Сохраняем энкодер на Диск для будущего деплоя в Streamlit
import os
os.makedirs('/content/drive/MyDrive/nlp_project', exist_ok=True)
joblib.dump(label_encoder, '/content/drive/MyDrive/nlp_project/label_encoder.pkl')

# Разбиваем на обучение и валидацию (80/20) с сохранением пропорций классов
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(f"Обучающая выборка: {len(train_df)} строк")
print(f"Валидационная выборка: {len(val_df)} строк")


Обучающая выборка: 2281 строк
Валидационная выборка: 571 строк


In [5]:
from transformers import AutoTokenizer
from datasets import Dataset

model_name = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # Токенизируем колонку 'content'
    return tokenizer(examples['content'], truncation=True, padding='max_length', max_length=256)

# Переводим в формат Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df[['content', 'label']])
val_dataset = Dataset.from_pandas(val_df[['content', 'label']])

# Запускаем маппинг токенизатора
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.74M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/2281 [00:00<?, ? examples/s]

Map:   0%|          | 0/571 [00:00<?, ? examples/s]

In [7]:
import numpy as np
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score, accuracy_score

num_labels = len(label_encoder.classes_)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    f1 = f1_score(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1_macro': f1}

training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5, # 5 эпох будет оптимально для такого объема
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    report_to='none',
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# Запуск обучения (убедитесь, что в Colab включен GPU: Изменить -> Настройки блокнота -> T4 GPU)
trainer.train()

# Сохраняем готовую модель на Диск
model.save_pretrained('/content/drive/MyDrive/nlp_project/best_model')
tokenizer.save_pretrained('/content/drive/MyDrive/nlp_project/best_model')
print("Модель успешно сохранена на Google Диск!")


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.464377,0.396913,0.926445,0.928015
2,0.165780,0.223363,0.943958,0.945109
3,0.097948,0.189278,0.949212,0.950322
4,0.045826,0.173445,0.949212,0.950643
5,0.027784,0.174571,0.949212,0.950491


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель успешно сохранена на Google Диск!


In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import joblib
import time

# 1. Загружаем сохраненные артефакты
model_path = '/content/drive/MyDrive/nlp_project/best_model'
encoder_path = '/content/drive/MyDrive/nlp_project/label_encoder.pkl'

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
label_encoder = joblib.load(encoder_path)

# Переводим модель в режим оценки
model.eval()

# 2. Функция для предсказания категории
def predict_category(text):
    start_time = time.time()

    # Токенизация текста
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256)

    # Получение предсказания без расчета градиентов
    with torch.no_grad():
        outputs = model(**inputs)

    # Расчет вероятностей и выбор лучшего класса
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    pred_class_idx = torch.argmax(probs, dim=-1).item()
    confidence = probs[0][pred_class_idx].item()

    # Перевод индекса обратно в текстовое название
    predicted_label = label_encoder.inverse_transform([pred_class_idx])[0]

    inference_time = (time.time() - start_time) * 1000  # Переводим в мс

    print(f"Текст: {text[:60]}...")
    print(f"Результат: {predicted_label} (Уверенность: {confidence:.2%})")
    print(f"Время инференса: {inference_time:.2f} мс")
    print("-" * 50)

# 3. Список тестовых фраз для проверки всех 5 ваших классов
test_phrases = [
    "Биткоин пробил очередной исторический максимум на фоне новостей от ФРС.",
    "Вчера на неделе моды в Париже бренд показал новую коллекцию оверсайз пиджаков.",
    "Сборная выиграла золотую медаль в финальном матче, обыграв соперников со счетом 3:1.",
    "OpenAI представила новую языковую модель, которая работает в 10 раз быстрее.",
    "Центральный банк повысил ключевую ставку, из-за чего изменятся условия по кредитам."
]

# Запуск тестов
for phrase in test_phrases:
    predict_category(phrase)


Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

Текст: Биткоин пробил очередной исторический максимум на фоне новос...
Результат: крипта (Уверенность: 97.00%)
Время инференса: 213.29 мс
--------------------------------------------------
Текст: Вчера на неделе моды в Париже бренд показал новую коллекцию ...
Результат: мода (Уверенность: 98.22%)
Время инференса: 28.05 мс
--------------------------------------------------
Текст: Сборная выиграла золотую медаль в финальном матче, обыграв с...
Результат: спорт (Уверенность: 97.75%)
Время инференса: 17.26 мс
--------------------------------------------------
Текст: OpenAI представила новую языковую модель, которая работает в...
Результат: технологии (Уверенность: 95.88%)
Время инференса: 21.24 мс
--------------------------------------------------
Текст: Центральный банк повысил ключевую ставку, из-за чего изменят...
Результат: финансы (Уверенность: 90.67%)
Время инференса: 18.80 мс
--------------------------------------------------
